In [7]:
import pandas as pd
from os import scandir
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt
import pickle
# custom
import custom as cst

# Thresholding - HSV

In [2]:
thresh = {}

In [3]:
low, high, thresh = cst.threshing('data/MEL/ISIC_0014800_downsampled.jpg', 'HSV', thresh=thresh)

# Thresholding - LAB

In [8]:
with open('thresh.pickle', 'rb') as fin: thresh = pickle.load(fin)

In [12]:
low, high, thresh = cst.threshing('data/MEL/ISIC_0014366_downsampled.jpg', 'LAB', thresh=thresh, low=low, high=high)

In [11]:
low = thresh['data/NV/ISIC_0008025_downsampled.jpg']['LOW']
high = thresh['data/NV/ISIC_0008025_downsampled.jpg']['HIGH']

In [10]:
thresh

{'data/MEL/ISIC_0014800_downsampled.jpg': {'LOW': (30, 155, 150),
  'HIGH': (160, 255, 180)},
 'data/MEL/ISIC_0014912_downsampled.jpg': {'LOW': (0, 131, 130),
  'HIGH': (153, 176, 170)},
 'data/NV/ISIC_0000024_downsampled.jpg': {'LOW': (0, 131, 130),
  'HIGH': (170, 176, 170)},
 'data/NV/ISIC_0000057_downsampled.jpg': {'LOW': (0, 131, 130),
  'HIGH': (170, 176, 170)},
 'data/NV/ISIC_0008025_downsampled.jpg': {'LOW': (0, 131, 149),
  'HIGH': (170, 176, 171)},
 'data/MEL/ISIC_0014366_downsampled.jpg': {'LOW': (0, 0, 0),
  'HIGH': (0, 0, 0)}}

# KMeans

In [27]:
img = cv.imread('data/NV/ISIC_0007796_downsampled.jpg')
show(img)
Z = img.reshape((-1, 3)).astype(np.float32)
img = cv.cvtColor(cv.imread('data/NV/ISIC_0007796_downsampled.jpg'), cv.COLOR_BGR2HSV)
prova = cv.inRange(img, (0, 100, 16), (17, 255, 200))

# define criteria, number of clusters(K) and apply kmeans()
criteria = (cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER, 10, 1.0)
K = 2
_, label, center = cv.kmeans(Z,K,None,criteria,10, cv.KMEANS_PP_CENTERS)
 
# Now convert back into uint8, and make original image
center = np.uint8(center)

# darker region
arg_dark = np.argmin(center[..., 0])
flat_idx = np.nonzero(label.flatten() == arg_dark)

# mask for image
mask = np.zeros(img.shape[0] * img.shape[1], dtype=np.uint8)
mask[flat_idx] = 255
mask = mask.reshape(img.shape[:2])

masked_img = cv.bitwise_and(img, img, mask=mask)

# bilateral filter
bil = cv.bilateralFilter(masked_img, d=9, sigmaColor=10, sigmaSpace=39)
# masked_img = cv.cvtColor(masked_img, cv.COLOR_BGR2GRAY)
masked_img = cv.cvtColor(masked_img, cv.COLOR_BGR2LAB)
bil = cv.cvtColor(bil, cv.COLOR_BGR2LAB)
# create a CLAHE object (Arguments are optional).
clahe = cv.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
cl1 = clahe.apply(masked_img[..., 0])
cl2 = clahe.apply(bil[..., 0])
bil[..., 0] = cl2
masked_img[..., 0] = cl1

# out = np.hstack((masked_img, cl1))
# show(out)
show(cv.cvtColor(masked_img, cv.COLOR_LAB2BGR))
show(cv.cvtColor(bil, cv.COLOR_LAB2BGR))

## normal shit

In [28]:
img = cv.imread('data/NV/ISIC_0007796_downsampled.jpg')
show(img)
Z = img.reshape((-1, 3)).astype(np.float32)

# define criteria, number of clusters(K) and apply kmeans()
criteria = (cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER, 10, 1.0)
K = 2
_, label, center = cv.kmeans(Z,K,None,criteria,10, cv.KMEANS_PP_CENTERS)
 
# Now convert back into uint8, and make original image
center = np.uint8(center)

# darker region
arg_dark = np.argmin(center[..., 0])
flat_idx = np.nonzero(label.flatten() == arg_dark)

# mask for image
mask = np.zeros(img.shape[0] * img.shape[1], dtype=np.uint8)
mask[flat_idx] = 255
mask = mask.reshape(img.shape[:2])

masked_img = cv.bitwise_and(img, img, mask=mask)

# bilateral filter
bil = cv.bilateralFilter(masked_img, d=9, sigmaColor=10, sigmaSpace=39)
# masked_img = cv.cvtColor(masked_img, cv.COLOR_BGR2GRAY)
masked_img = cv.cvtColor(masked_img, cv.COLOR_BGR2LAB)
bil = cv.cvtColor(bil, cv.COLOR_BGR2LAB)
# create a CLAHE object (Arguments are optional).
clahe = cv.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
cl1 = clahe.apply(masked_img[..., 0])
cl2 = clahe.apply(bil[..., 0])
bil[..., 0] = cl2
masked_img[..., 0] = cl1

# out = np.hstack((masked_img, cl1))
# show(out)
show(cv.cvtColor(masked_img, cv.COLOR_LAB2BGR))
show(cv.cvtColor(bil, cv.COLOR_LAB2BGR))

In [27]:
show(masked_img)

In [37]:
bil = cv.bilateralFilter(masked_img, d=9, sigmaColor=10, sigmaSpace=39)
show(np.hstack((masked_img, bil)))